# SkillBridge by Zequa

## Day 3: Data Cleaning and Standardization

### Objective
Clean, standardize, and integrate global occupation
and skill datasets from ESCO and O*NET.

### Main Activities
1. Load raw datasets
2. Inspect missing values and duplicates
3. Clean occupation and skill records
4. Normalize job titles and skill names
5. Standardize identifiers
6. Integrate occupation-skill relationships
7. Build a unified global occupation database
8. Validate and export processed datasets

### Data Sources
- ESCO v1.2.1
- O*NET 31.0

### Technologies
- Python
- Pandas
- NumPy
- Regular Expressions
- Scikit-learn

In [1]:
import os
import re
import sys
import pandas as pd
import numpy as np

from pathlib import Path

print("Python Version:", sys.version)
print("Pandas Version:", pd.__version__)
print("NumPy Version:", np.__version__)

print("\nSkillBridge Day 3 Notebook Ready!")

Python Version: 3.12.10 (v3.12.10:0cc81280367, Apr  8 2025, 08:47:00) [Clang 13.0.0 (clang-1300.0.29.30)]
Pandas Version: 2.2.3
NumPy Version: 1.26.4

SkillBridge Day 3 Notebook Ready!


In [2]:
# SkillBridge Day 3 - Dataset Loading

# Project paths
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
ESCO_PATH = RAW_DATA_PATH / "esco"
ONET_PATH = RAW_DATA_PATH / "onet"

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

# Load ESCO datasets
esco_occupations = pd.read_csv(
    ESCO_PATH / "occupations_en.csv"
)

esco_skills = pd.read_csv(
    ESCO_PATH / "skills_en.csv"
)

esco_relations = pd.read_csv(
    ESCO_PATH / "occupationSkillRelations_en.csv"
)

esco_skill_groups = pd.read_csv(
    ESCO_PATH / "skillGroups_en.csv"
)

# Load O*NET datasets
onet_occupations = pd.read_csv(
    ONET_PATH / "occupation_data.csv"
)

onet_skills = pd.read_csv(
    ONET_PATH / "essential_skills.csv"
)

onet_job_titles = pd.read_csv(
    ONET_PATH / "job_titles.csv"
)

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [3]:
# Dataset overview

datasets = {
    "ESCO Occupations": esco_occupations,
    "ESCO Skills": esco_skills,
    "ESCO Relationships": esco_relations,
    "ESCO Skill Groups": esco_skill_groups,
    "O*NET Occupations": onet_occupations,
    "O*NET Skills": onet_skills,
    "O*NET Job Titles": onet_job_titles
}

dataset_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    }
    for name, df in datasets.items()
])

display(dataset_summary)

print("\nTotal records:",
      dataset_summary["Rows"].sum())

,Dataset,Rows,Columns
0,ESCO Occupations,3043,15
1,ESCO Skills,13960,13
2,ESCO Relationships,126051,6
3,ESCO Skill Groups,640,11
4,O*NET Occupations,1016,3
5,O*NET Skills,18200,15
6,O*NET Job Titles,54269,5



Total records: 217179


In [4]:
# Data Quality Inspection Function

def inspect_data_quality(df, dataset_name):

    print("\n" + "=" * 60)
    print(f"DATA QUALITY REPORT: {dataset_name}")
    print("=" * 60)

    print("Total Rows:", len(df))
    print("Total Columns:", len(df.columns))

    print("Duplicate Rows:", df.duplicated().sum())

    print("Missing Cells:", df.isnull().sum().sum())

    print("\nMissing Values by Column:")

    missing = df.isnull().sum()

    missing_report = pd.DataFrame({
        "Column": missing.index,
        "Missing Values": missing.values,
        "Missing Percentage": (
            missing.values / len(df) * 100
        ).round(2)
    })

    display(
        missing_report[
            missing_report["Missing Values"] > 0
        ]
    )

    print("\nData Types:")
    display(df.dtypes.to_frame("Data Type"))


# Inspect all datasets

for name, df in datasets.items():

    inspect_data_quality(df, name)


DATA QUALITY REPORT: ESCO Occupations
Total Rows: 3043
Total Columns: 15
Duplicate Rows: 0
Missing Cells: 8803

Missing Values by Column:


,Column,Missing Values,Missing Percentage
5,hiddenLabels,3035,99.74
9,scopeNote,2733,89.81
10,definition,3035,99.74



Data Types:


,Data Type
conceptType,object
conceptUri,object
iscoGroup,int64
preferredLabel,object
altLabels,object
hiddenLabels,object
status,object
modifiedDate,object
regulatedProfessionNote,object
scopeNote,object



DATA QUALITY REPORT: ESCO Skills
Total Rows: 13960
Total Columns: 13
Duplicate Rows: 0
Missing Cells: 41517

Missing Values by Column:


,Column,Missing Values,Missing Percentage
2,skillType,5,0.04
3,reuseLevel,5,0.04
5,altLabels,18,0.13
6,hiddenLabels,13807,98.90
9,scopeNote,13724,98.31
10,definition,13958,99.99


... (18 outputs left out)


DATA QUALITY REPORT: O*NET Job Titles
Total Rows: 54269
Total Columns: 5
Duplicate Rows: 0
Missing Cells: 49814

Missing Values by Column:


,Column,Missing Values,Missing Percentage
3,Short Title,49814,91.79



Data Types:


,Data Type
O*NET-SOC Code,object
Title,object
Job Title,object
Short Title,object
Source(s),object


In [5]:
# Inspect ESCO occupation schema

print("ESCO Occupation Columns:")
print(esco_occupations.columns.tolist())

display(esco_occupations.head(5))

ESCO Occupation Columns:
['conceptType', 'conceptUri', 'iscoGroup', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'regulatedProfessionNote', 'scopeNote', 'definition', 'inScheme', 'description', 'code', 'naceCode']


,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,naceCode
0,Occupation,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,director of technical arts\ntechnical supervis...,NaN,released,2024-01-25T11:28:50.295Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Technical directors realise the artistic visio...,2654.1.7,http://data.europa.eu/ux2/nace2.1/9031
1,Occupation,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,wire drawer\nforming machine operative\ndraw m...,NaN,released,2024-01-23T10:09:32.099Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Metal drawing machine operators set up and ope...,8121.4,http://data.europa.eu/ux2/nace2.1/242
2,Occupation,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,precision device quality control supervisor\np...,NaN,released,2024-01-25T15:00:12.188Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Precision device inspectors make sure precisio...,7543.10.3,http://data.europa.eu/ux2/nace2.1/2651
3,Occupation,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,air traffic safety electronics hardware specia...,NaN,released,2024-01-29T16:01:13.998Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Air traffic safety technicians provide technic...,3155.1,http://data.europa.eu/ux2/nace2.1/5223
4,Occupation,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,yield manager\nhospitality yields manager\nhos...,NaN,released,2024-01-11T10:28:45.871Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Hospitality revenue managers maximise revenue ...,2431.9,"http://data.europa.eu/ux2/nace2.1/701,\nhttp:/..."


In [6]:
# Create a working copy

esco_occupations_clean = esco_occupations.copy()

# Standardize column names
esco_occupations_clean.columns = (
    esco_occupations_clean.columns
    .str.strip()
)

# Remove leading and trailing whitespace from text fields
text_columns = esco_occupations_clean.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    esco_occupations_clean[column] = (
        esco_occupations_clean[column]
        .astype("string")
        .str.strip()
    )

# Remove records without an occupation identifier
# or preferred occupation title.
esco_occupations_clean = (
    esco_occupations_clean.dropna(
        subset=["conceptUri", "preferredLabel"]
    )
)

# Remove empty occupation identifiers or titles
esco_occupations_clean = (
    esco_occupations_clean[
        esco_occupations_clean["conceptUri"].ne("")
        & esco_occupations_clean["preferredLabel"].ne("")
    ]
)

# Remove exact duplicate records
esco_occupations_clean = (
    esco_occupations_clean.drop_duplicates()
)

# Reset index
esco_occupations_clean = (
    esco_occupations_clean.reset_index(drop=True)
)

print("ESCO occupation cleaning completed!")

print("\nOriginal records:", len(esco_occupations))
print("Cleaned records:", len(esco_occupations_clean))
print("Removed records:",
      len(esco_occupations) - len(esco_occupations_clean))

display(esco_occupations_clean.head(10))

ESCO occupation cleaning completed!

Original records: 3043
Cleaned records: 3043
Removed records: 0


,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,naceCode
0,Occupation,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,director of technical arts technical superviso...,<NA>,released,2024-01-25T11:28:50.295Z,http://data.europa.eu/esco/regulated-professio...,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/memb...,Technical directors realise the artistic visio...,2654.1.7,http://data.europa.eu/ux2/nace2.1/9031
1,Occupation,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,wire drawer forming machine operative draw mac...,<NA>,released,2024-01-23T10:09:32.099Z,http://data.europa.eu/esco/regulated-professio...,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/memb...,Metal drawing machine operators set up and ope...,8121.4,http://data.europa.eu/ux2/nace2.1/242
2,Occupation,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,precision device quality control supervisor pr...,<NA>,released,2024-01-25T15:00:12.188Z,http://data.europa.eu/esco/regulated-professio...,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/memb...,Precision device inspectors make sure precisio...,7543.10.3,http://data.europa.eu/ux2/nace2.1/2651
3,Occupation,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,air traffic safety electronics hardware specia...,<NA>,released,2024-01-29T16:01:13.998Z,http://data.europa.eu/esco/regulated-professio...,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/memb...,Air traffic safety technicians provide technic...,3155.1,http://data.europa.eu/ux2/nace2.1/5223
4,Occupation,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,yield manager hospitality yields manager hospi...,<NA>,released,2024-01-11T10:28:45.871Z,http://data.europa.eu/esco/regulated-professio...,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/memb...,Hospitality revenue managers maximise revenue ...,2431.9,"http://data.europa.eu/ux2/nace2.1/701, http://..."
5,Occupation,http://data.europa.eu/esco/occupation/0044c991...,3212,medical laboratory assistant,biomedical laboratory research assistant clini...,<NA>,released,2024-01-24T15:38:45.842Z,http://data.europa.eu/esco/regulated-professio...,Excludes people performing analysis of blood s...,<NA>,http://data.europa.eu/esco/concept-scheme/memb...,Medical laboratory assistants work under super...,3212.2,http://data.europa.eu/ux2/nace2.1/861
6,Occupation,http://data.europa.eu/esco/occupation/00634fc4...,3116,asphalt laboratory technician,bitumen technician asphalt production technica...,<NA>,released,2024-01-26T15:03:25.743Z,http://data.europa.eu/esco/regulated-professio...,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/memb...,Asphalt laboratory technicians perform asphalt...,3116.1.1,"http://data.europa.eu/ux2/nace2.1/712, http://..."
7,Occupation,http://data.europa.eu/esco/occupation/00674f21...,5312,primary school teaching assistant,primary school TA primary school assistant pri...,<NA>,released,2023-07-04T08:19:38.994Z,http://data.europa.eu/esco/regulated-professio...,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/memb...,Primary school teaching assistants provide ins...,5312.2,http://data.europa.eu/ux2/nace2.1/852
8,Occupation,http://data.europa.eu/esco/occupation/006cc1f9...,2264,physiotherapist,respiratory therapist masseuse rehabilitation ...,<NA>,released,2024-01-16T10:48:48.922Z,http://data.europa.eu/esco/regulated-professio...,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/memb...,Physiotherapists are autonomous health profess...,2264.2,http://data.europa.eu/ux2/nace2.1/8695
9,Occupation,http://data.europa.eu/esco/occupation/00747307...,2310,performing arts theatre instructor,professor of theatre acting school teacher the...,<NA>,released,2024-01-16T13:49:12.7Z,http://data.europa.eu/esco/regulated-professio...,<N

In [7]:
print("ESCO Skill Columns:")
print(esco_skills.columns.tolist())

display(esco_skills.head(5))

ESCO Skill Columns:
['conceptType', 'conceptUri', 'skillType', 'reuseLevel', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'scopeNote', 'definition', 'inScheme', 'description']


,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,scopeNote,definition,inScheme,description
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,manage musical staff,manage music staff\ncoordinate duties of music...,NaN,released,2023-11-30T15:53:37.136Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Assign and manage staff tasks in areas such as...
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,supervise correctional procedures,manage prison procedures\nmonitor correctional...,NaN,released,2023-11-30T15:04:00.689Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Supervise the operations of a correctional fac...
2,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000709ed-2be5...,skill/competence,sector-specific,apply anti-oppressive practices,make use of anti-oppressive practices\nuse ant...,NaN,released,2023-11-28T10:45:53.54Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Identify oppression in societies, economies, c..."
3,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0007bdc2-dd15...,skill/competence,sector-specific,control compliance of railway vehicles regulat...,checking compliance with rolling stock regulat...,NaN,released,2023-11-30T16:29:18.273Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Inspect rolling stock, components and systems ..."
4,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00090cc1-1f27...,skill/competence,cross-sector,identify available services,determine available services\nclassify availab...,NaN,released,2023-11-28T10:38:49.206Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Identify the different services available for ...


In [8]:
# Create a working copy
esco_skills_clean = esco_skills.copy()

# Trim column names
esco_skills_clean.columns = (
    esco_skills_clean.columns.str.strip()
)

# Clean text fields without converting missing values into strings
text_columns = esco_skills_clean.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    esco_skills_clean[column] = (
        esco_skills_clean[column]
        .astype("string")
        .str.strip()
    )

# Remove records without essential identifiers or labels
esco_skills_clean = esco_skills_clean.dropna(
    subset=["conceptUri", "preferredLabel"]
)

esco_skills_clean = esco_skills_clean[
    esco_skills_clean["conceptUri"].ne("")
    & esco_skills_clean["preferredLabel"].ne("")
]

# Remove exact duplicate rows
esco_skills_clean = esco_skills_clean.drop_duplicates()

# Reset index
esco_skills_clean = esco_skills_clean.reset_index(drop=True)

print("ESCO skill cleaning completed!")

print("\nOriginal records:", len(esco_skills))
print("Cleaned records:", len(esco_skills_clean))
print(
    "Removed records:",
    len(esco_skills) - len(esco_skills_clean)
)

display(esco_skills_clean.head(10))

ESCO skill cleaning completed!

Original records: 13960
Cleaned records: 13960
Removed records: 0


,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,scopeNote,definition,inScheme,description
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,manage musical staff,manage music staff coordinate duties of musica...,<NA>,released,2023-11-30T15:53:37.136Z,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/skil...,Assign and manage staff tasks in areas such as...
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,supervise correctional procedures,manage prison procedures monitor correctional ...,<NA>,released,2023-11-30T15:04:00.689Z,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/skil...,Supervise the operations of a correctional fac...
2,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000709ed-2be5...,skill/competence,sector-specific,apply anti-oppressive practices,make use of anti-oppressive practices use anti...,<NA>,released,2023-11-28T10:45:53.54Z,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/skil...,"Identify oppression in societies, economies, c..."
3,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0007bdc2-dd15...,skill/competence,sector-specific,control compliance of railway vehicles regulat...,checking compliance with rolling stock regulat...,<NA>,released,2023-11-30T16:29:18.273Z,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/skil...,"Inspect rolling stock, components and systems ..."
4,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00090cc1-1f27...,skill/competence,cross-sector,identify available services,determine available services classify availabl...,<NA>,released,2023-11-28T10:38:49.206Z,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/skil...,Identify the different services available for ...
5,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000bb1e4-89f0...,skill/competence,cross-sector,perform toxicological studies,carry out toxicological studies apply toxicolo...,<NA>,released,2023-11-30T15:10:42.752Z,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/skil...,Perform tests to detect poisons or drug misuse...
6,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000c94d2-2a2e...,skill/competence,occupation-specific,ensure coquille uniformity,ensuring coquille is uniform make sure coquill...,<NA>,released,2023-11-30T16:49:26.078Z,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/skil...,Oversee uniformity of coquilles; use casting e...
7,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000f1d3d-220f...,knowledge,sector-specific,Haskell,Haskell techniques,<NA>,released,2023-11-30T16:13:21.77Z,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/skil...,The techniques and principles of software deve...
8,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/001115fb-569f...,skill/competence,transversal,show initiative,initiate action be a driving force give impetu...,active participation activity initiative great...,released,2021-09-02T16:37:00.315Z,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/skil...,Be proactive and take the first step in an act...
9,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/001d46db-035e...,skill/competence,sector-specific,train staff to reduce food waste,teach students food waste reduction practices ...,<NA>,released,2021-01-20T14:31:01.524Z,<NA>,<NA>,http://data.europa.eu/esco/concept-scheme/skil...,Establish new trainings and staff development ...


In [9]:
print("O*NET Occupation Columns:")
print(onet_occupations.columns.tolist())

display(onet_occupations.head(5))

O*NET Occupation Columns:
['O*NET-SOC Code', 'Title', 'Description']


,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."


In [10]:
# Create working copy
onet_occupations_clean = onet_occupations.copy()

# Clean column names
onet_occupations_clean.columns = (
    onet_occupations_clean.columns.str.strip()
)

# Clean text fields
text_columns = onet_occupations_clean.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    onet_occupations_clean[column] = (
        onet_occupations_clean[column]
        .astype("string")
        .str.strip()
    )

# Remove records without occupation code or title
onet_occupations_clean = onet_occupations_clean.dropna(
    subset=["O*NET-SOC Code", "Title"]
)

onet_occupations_clean = onet_occupations_clean[
    onet_occupations_clean["O*NET-SOC Code"].ne("")
    & onet_occupations_clean["Title"].ne("")
]

# Remove exact duplicate records
onet_occupations_clean = (
    onet_occupations_clean.drop_duplicates()
)

# Reset index
onet_occupations_clean = (
    onet_occupations_clean.reset_index(drop=True)
)

print("O*NET occupation cleaning completed!")

print("\nOriginal records:", len(onet_occupations))
print("Cleaned records:", len(onet_occupations_clean))
print(
    "Removed records:",
    len(onet_occupations) - len(onet_occupations_clean)
)

display(onet_occupations_clean.head(10))

O*NET occupation cleaning completed!

Original records: 1016
Cleaned records: 1016
Removed records: 0


,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."
5,11-2021.00,Marketing Managers,"Plan, direct, or coordinate marketing policies..."
6,11-2022.00,Sales Managers,"Plan, direct, or coordinate the actual distrib..."
7,11-2032.00,Public Relations Managers,"Plan, direct, or coordinate activities designe..."
8,11-2033.00,Fundraising Managers,"Plan, direct, or coordinate activities to soli..."
9,11-3012.00,Administrative Services Managers,"Plan, direct, or coordinate one or more admini..."


In [11]:
# Create O*NET skills working copy

onet_skills_clean = onet_skills.copy()

# Clean column names
onet_skills_clean.columns = (
    onet_skills_clean.columns.str.strip()
)

# Clean text columns
text_columns = onet_skills_clean.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    onet_skills_clean[column] = (
        onet_skills_clean[column]
        .astype("string")
        .str.strip()
    )

print("O*NET skills working copy created!")
print("Total records:", len(onet_skills_clean))

display(onet_skills_clean.head(5))

O*NET skills working copy created!
Total records: 18200


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,<NA>,08/2023,Analyst
1,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,LV,Level,4.62,8,0.1830,4.2664,4.9836,N,N,08/2023,Analyst
2,11-1011.00,Chief Executives,2.A.1.b,Active Listening,IM,Importance,4.00,8,0.0000,4.0000,4.0000,N,<NA>,08/2023,Analyst
3,11-1011.00,Chief Executives,2.A.1.b,Active Listening,LV,Level,4.75,8,0.1637,4.4292,5.0708,N,N,08/2023,Analyst
4,11-1011.00,Chief Executives,2.A.1.c,Writing,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,<NA>,08/2023,Analyst


In [12]:
# Remove records missing essential identifiers

required_columns = [
    "O*NET-SOC Code",
    "Element ID",
    "Element Name",
    "Scale ID",
    "Data Value"
]

onet_skills_clean = onet_skills_clean.dropna(
    subset=required_columns
)

# Convert Data Value into numeric format
onet_skills_clean["Data Value"] = pd.to_numeric(
    onet_skills_clean["Data Value"],
    errors="coerce"
)

# Remove invalid numeric measurements
onet_skills_clean = onet_skills_clean.dropna(
    subset=["Data Value"]
)

# Remove exact duplicate records
onet_skills_clean = onet_skills_clean.drop_duplicates()

# Reset index
onet_skills_clean = onet_skills_clean.reset_index(drop=True)

print("O*NET skill cleaning completed!")

print("\nOriginal records:", len(onet_skills))
print("Cleaned records:", len(onet_skills_clean))
print(
    "Removed records:",
    len(onet_skills) - len(onet_skills_clean)
)

print("\nScale distribution:")
display(
    onet_skills_clean["Scale ID"].value_counts()
)

display(onet_skills_clean.head(10))

O*NET skill cleaning completed!

Original records: 18200
Cleaned records: 18200
Removed records: 0

Scale distribution:


Scale ID
IM    9100
LV    9100
Name: count, dtype: Int64

,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,<NA>,08/2023,Analyst
1,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,LV,Level,4.62,8,0.1830,4.2664,4.9836,N,N,08/2023,Analyst
2,11-1011.00,Chief Executives,2.A.1.b,Active Listening,IM,Importance,4.00,8,0.0000,4.0000,4.0000,N,<NA>,08/2023,Analyst
3,11-1011.00,Chief Executives,2.A.1.b,Active Listening,LV,Level,4.75,8,0.1637,4.4292,5.0708,N,N,08/2023,Analyst
4,11-1011.00,Chief Executives,2.A.1.c,Writing,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,<NA>,08/2023,Analyst
5,11-1011.00,Chief Executives,2.A.1.c,Writing,LV,Level,4.38,8,0.1830,4.0164,4.7336,N,N,08/2023,Analyst
6,11-1011.00,Chief Executives,2.A.1.d,Speaking,IM,Importance,4.25,8,0.1637,3.9292,4.5708,N,<NA>,08/2023,Analyst
7,11-1011.00,Chief Executives,2.A.1.d,Speaking,LV,Level,4.75,8,0.1637,4.4292,5.0708,N,N,08/2023,Analyst
8,11-1011.00,Chief Executives,2.A.1.e,Mathematics,IM,Importance,3.25,8,0.2500,2.7600,3.7400,N,<NA>,08/2023,Analyst
9,11-1011.00,Chief Executives,2.A.1.e,Mathematics,LV,Level,3.50,8,0.2673,2.9762,4.0238,N,N,08/2023,Analyst


In [13]:
def normalize_job_title(title):
    if pd.isna(title):
        return None

    title = str(title).strip().lower()

    # Replace multiple spaces with one space
    title = re.sub(r"\s+", " ", title)

    # Remove unnecessary punctuation
    title = re.sub(r"[^\w\s-]", "", title)

    return title


print(normalize_job_title(" Data Scientist "))
print(normalize_job_title("DATA ANALYST"))
print(normalize_job_title("Machine Learning Engineer"))

data scientist
data analyst
machine learning engineer


In [14]:
# Recreate cleaned O*NET job titles dataset

onet_job_titles_clean = onet_job_titles.copy()

# Remove whitespace from column names
onet_job_titles_clean.columns = (
    onet_job_titles_clean.columns.str.strip()
)

# Clean text columns
for column in onet_job_titles_clean.select_dtypes(
    include=["object", "string"]
).columns:
    onet_job_titles_clean[column] = (
        onet_job_titles_clean[column]
        .astype("string")
        .str.strip()
    )

# Remove records with missing essential fields
required_columns = [
    "O*NET-SOC Code",
    "Job Title"
]

onet_job_titles_clean = onet_job_titles_clean.dropna(
    subset=required_columns
)

# Remove blank titles and occupation codes
onet_job_titles_clean = onet_job_titles_clean[
    (onet_job_titles_clean["Job Title"] != "") &
    (onet_job_titles_clean["O*NET-SOC Code"] != "")
]

# Remove exact duplicate records
onet_job_titles_clean = (
    onet_job_titles_clean.drop_duplicates().reset_index(drop=True)
)

print("O*NET job titles dataset prepared!")
print("Total records:", len(onet_job_titles_clean))

display(onet_job_titles_clean.head())

O*NET job titles dataset prepared!
Total records: 54269


,O*NET-SOC Code,Title,Job Title,Short Title,Source(s)
0,11-1011.00,Chief Executives,Aeronautics Commission Director,<NA>,08
1,11-1011.00,Chief Executives,Agency Owner,<NA>,10
2,11-1011.00,Chief Executives,Agricultural Services Director,<NA>,08
3,11-1011.00,Chief Executives,Arts and Humanities Council Director,<NA>,08
4,11-1011.00,Chief Executives,Bank President,<NA>,09


In [15]:
# Create a copy to preserve the original dataset
onet_job_titles_clean = onet_job_titles_clean.copy()

# Preserve original titles and create normalized titles
onet_job_titles_clean["Normalized Job Title"] = (
    onet_job_titles_clean["Job Title"]
    .apply(normalize_job_title)
)

display(
    onet_job_titles_clean[
        ["Job Title", "Normalized Job Title"]
    ].head(15)
)

,Job Title,Normalized Job Title
0,Aeronautics Commission Director,aeronautics commission director
1,Agency Owner,agency owner
2,Agricultural Services Director,agricultural services director
3,Arts and Humanities Council Director,arts and humanities council director
4,Bank President,bank president
5,Bureau Chief,bureau chief
6,Business Development Executive (BD Executive),business development executive bd executive
7,Business Development Officer (BD Officer),business development officer bd officer
8,Business Enterprise Officer,business enterprise officer
9,Business Executive,business executive


In [16]:
def normalize_skill_name(skill):
    if pd.isna(skill):
        return None

    skill = str(skill).strip().lower()
    skill = re.sub(r"\s+", " ", skill)
    skill = re.sub(r"[^\w\s+#.-]", "", skill)

    return skill


# Test examples
test_skills = [
    "Python Programming",
    " MACHINE LEARNING ",
    "Data Analysis",
    "SQL & Database Management"
]

for skill in test_skills:
    print(skill, "→", normalize_skill_name(skill))

Python Programming → python programming
 MACHINE LEARNING  → machine learning
Data Analysis → data analysis
SQL & Database Management → sql  database management


In [17]:
esco_skills_clean = esco_skills_clean.copy()

esco_skills_clean["Normalized Skill"] = (
    esco_skills_clean["preferredLabel"]
    .apply(normalize_skill_name)
)

display(
    esco_skills_clean[
        ["preferredLabel", "Normalized Skill"]
    ].head(15)
)

,preferredLabel,Normalized Skill
0,manage musical staff,manage musical staff
1,supervise correctional procedures,supervise correctional procedures
2,apply anti-oppressive practices,apply anti-oppressive practices
3,control compliance of railway vehicles regulat...,control compliance of railway vehicles regulat...
4,identify available services,identify available services
5,perform toxicological studies,perform toxicological studies
6,ensure coquille uniformity,ensure coquille uniformity
7,Haskell,haskell
8,show initiative,show initiative
9,train staff to reduce food waste,train staff to reduce food waste


In [18]:
# Create a copy
esco_occupations_clean = esco_occupations_clean.copy()

# Standardize ESCO occupation URI
esco_occupations_clean["Occupation ID"] = (
    esco_occupations_clean["conceptUri"]
    .astype("string")
    .str.strip()
)

# Check results
print("ESCO occupation IDs:")
display(
    esco_occupations_clean[
        ["conceptUri", "Occupation ID", "preferredLabel"]
    ].head(10)
)

ESCO occupation IDs:


,conceptUri,Occupation ID,preferredLabel
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/occupation/00030d09...,technical director
1,http://data.europa.eu/esco/occupation/000e93a3...,http://data.europa.eu/esco/occupation/000e93a3...,metal drawing machine operator
2,http://data.europa.eu/esco/occupation/0019b951...,http://data.europa.eu/esco/occupation/0019b951...,precision device inspector
3,http://data.europa.eu/esco/occupation/0022f466...,http://data.europa.eu/esco/occupation/0022f466...,air traffic safety technician
4,http://data.europa.eu/esco/occupation/002da35b...,http://data.europa.eu/esco/occupation/002da35b...,hospitality revenue manager
5,http://data.europa.eu/esco/occupation/0044c991...,http://data.europa.eu/esco/occupation/0044c991...,medical laboratory assistant
6,http://data.europa.eu/esco/occupation/00634fc4...,http://data.europa.eu/esco/occupation/00634fc4...,asphalt laboratory technician
7,http://data.europa.eu/esco/occupation/00674f21...,http://data.europa.eu/esco/occupation/00674f21...,primary school teaching assistant
8,http://data.europa.eu/esco/occupation/006cc1f9...,http://data.europa.eu/esco/occupation/006cc1f9...,physiotherapist
9,http://data.europa.eu/esco/occupation/00747307...,http://data.europa.eu/esco/occupation/00747307...,performing arts theatre instructor


In [19]:
# Create a copy
onet_occupations_clean = onet_occupations_clean.copy()

# Standardize O*NET occupation codes
onet_occupations_clean["Occupation ID"] = (
    onet_occupations_clean["O*NET-SOC Code"]
    .astype("string")
    .str.strip()
)

print("O*NET occupation IDs:")
display(
    onet_occupations_clean[
        ["O*NET-SOC Code", "Occupation ID", "Title"]
    ].head(10)
)

O*NET occupation IDs:


,O*NET-SOC Code,Occupation ID,Title
0,11-1011.00,11-1011.00,Chief Executives
1,11-1011.03,11-1011.03,Chief Sustainability Officers
2,11-1021.00,11-1021.00,General and Operations Managers
3,11-1031.00,11-1031.00,Legislators
4,11-2011.00,11-2011.00,Advertising and Promotions Managers
5,11-2021.00,11-2021.00,Marketing Managers
6,11-2022.00,11-2022.00,Sales Managers
7,11-2032.00,11-2032.00,Public Relations Managers
8,11-2033.00,11-2033.00,Fundraising Managers
9,11-3012.00,11-3012.00,Administrative Services Managers


In [20]:
print("ESCO validation")
print(
    "Missing IDs:",
    esco_occupations_clean["Occupation ID"].isna().sum()
)
print(
    "Unique IDs:",
    esco_occupations_clean["Occupation ID"].nunique()
)

print("\nO*NET validation")
print(
    "Missing IDs:",
    onet_occupations_clean["Occupation ID"].isna().sum()
)
print(
    "Unique IDs:",
    onet_occupations_clean["Occupation ID"].nunique()
)

ESCO validation
Missing IDs: 0
Unique IDs: 3039

O*NET validation
Missing IDs: 0
Unique IDs: 1016


In [21]:
print("ESCO relationship columns:")
print(esco_relations.columns.tolist())

display(esco_relations.head(5))

ESCO relationship columns:
['occupationUri', 'occupationLabel', 'relationType', 'skillType', 'skillUri', 'skillLabel']


,occupationUri,occupationLabel,relationType,skillType,skillUri,skillLabel
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,knowledge,http://data.europa.eu/esco/skill/fed5b267-73fa...,theatre techniques
1,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/05bc7677-5a64...,organise rehearsals
2,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/271a36a0-bc7a...,write risk assessment on performing arts produ...
3,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/47ed1d37-971b...,coordinate with creative departments
4,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/591dd514-735b...,adapt to artists' creative demands


In [22]:
# Create working copy
esco_relations_clean = esco_relations.copy()

# Remove whitespace from column names
esco_relations_clean.columns = (
    esco_relations_clean.columns.str.strip()
)

# Clean text columns
for column in esco_relations_clean.select_dtypes(
    include=["object", "string"]
).columns:
    esco_relations_clean[column] = (
        esco_relations_clean[column]
        .astype("string")
        .str.strip()
    )

# Identify required relationship columns
required_columns = [
    "occupationUri",
    "skillUri"
]

missing_columns = [
    col for col in required_columns
    if col not in esco_relations_clean.columns
]

if missing_columns:
    print("Required columns not found:", missing_columns)
else:
    esco_relations_clean = (
        esco_relations_clean.dropna(subset=required_columns)
        .drop_duplicates()
        .reset_index(drop=True)
    )

    print("ESCO relationships cleaned!")
    print("Total relationships:", len(esco_relations_clean))

    display(esco_relations_clean.head(10))

ESCO relationships cleaned!
Total relationships: 126051


,occupationUri,occupationLabel,relationType,skillType,skillUri,skillLabel
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,knowledge,http://data.europa.eu/esco/skill/fed5b267-73fa...,theatre techniques
1,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/05bc7677-5a64...,organise rehearsals
2,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/271a36a0-bc7a...,write risk assessment on performing arts produ...
3,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/47ed1d37-971b...,coordinate with creative departments
4,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/591dd514-735b...,adapt to artists' creative demands
5,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/860be36a-d19b...,negotiate health and safety issues with third ...
6,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/892f8e2f-189a...,adapt designers’ work to the performance venue
7,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/93a68dcb-3dc6...,promote health and safety
8,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/f64fe2c2-d090...,coordinate technical teams in artistic product...
9,http://data.europa.eu/esco/occupation/00030d09...,technical director,optional,skill/competence,http://data.europa.eu/esco/skill/a2dfc063-7b24...,write technical riders


In [23]:
if all(
    col in esco_relations_clean.columns
    for col in ["occupationUri", "skillUri"]
):
    print(
        "Unique occupations:",
        esco_relations_clean["occupationUri"].nunique()
    )

    print(
        "Unique skills:",
        esco_relations_clean["skillUri"].nunique()
    )

    print(
        "Duplicate relationships:",
        esco_relations_clean.duplicated(
            subset=["occupationUri", "skillUri"]
        ).sum()
    )

Unique occupations: 3039
Unique skills: 13475
Duplicate relationships: 0


In [24]:
# Select relevant O*NET skill fields
onet_relationships = onet_skills_clean[
    [
        "O*NET-SOC Code",
        "Title",
        "Element ID",
        "Element Name",
        "Scale ID",
        "Scale Name",
        "Data Value"
    ]
].copy()

# Rename columns for consistency
onet_relationships = onet_relationships.rename(
    columns={
        "O*NET-SOC Code": "Occupation ID",
        "Title": "Occupation Name",
        "Element ID": "Skill ID",
        "Element Name": "Skill Name",
        "Scale ID": "Measurement ID",
        "Scale Name": "Measurement Name",
        "Data Value": "Skill Rating"
    }
)

display(onet_relationships.head(10))

,Occupation ID,Occupation Name,Skill ID,Skill Name,Measurement ID,Measurement Name,Skill Rating
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,IM,Importance,4.12
1,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,LV,Level,4.62
2,11-1011.00,Chief Executives,2.A.1.b,Active Listening,IM,Importance,4.00
3,11-1011.00,Chief Executives,2.A.1.b,Active Listening,LV,Level,4.75
4,11-1011.00,Chief Executives,2.A.1.c,Writing,IM,Importance,4.12
5,11-1011.00,Chief Executives,2.A.1.c,Writing,LV,Level,4.38
6,11-1011.00,Chief Executives,2.A.1.d,Speaking,IM,Importance,4.25
7,11-1011.00,Chief Executives,2.A.1.d,Speaking,LV,Level,4.75
8,11-1011.00,Chief Executives,2.A.1.e,Mathematics,IM,Importance,3.25
9,11-1011.00,Chief Executives,2.A.1.e,Mathematics,LV,Level,3.50


In [25]:
print("O*NET Relationship Summary")
print("=" * 40)

print("Total records:", len(onet_relationships))
print("Unique occupations:",
      onet_relationships["Occupation ID"].nunique())
print("Unique skills:",
      onet_relationships["Skill ID"].nunique())

print("\nMeasurement types:")
display(onet_relationships["Measurement Name"].value_counts())

O*NET Relationship Summary
Total records: 18200
Unique occupations: 910
Unique skills: 10

Measurement types:


Measurement Name
Importance    9100
Level         9100
Name: count, dtype: Int64

In [26]:
esco_unified = esco_occupations_clean[
    ["Occupation ID", "preferredLabel"]
].copy()

esco_unified = esco_unified.rename(
    columns={
        "preferredLabel": "Occupation Name"
    }
)

esco_unified["Source"] = "ESCO"


In [27]:
onet_unified = onet_occupations_clean[
    ["Occupation ID", "Title"]
].copy()

onet_unified = onet_unified.rename(
    columns={
        "Title": "Occupation Name"
    }
)

onet_unified["Source"] = "O*NET"

In [28]:
unified_occupations = pd.concat(
    [esco_unified, onet_unified],
    ignore_index=True
)

# Clean text fields
unified_occupations["Occupation ID"] = (
    unified_occupations["Occupation ID"]
    .astype("string")
    .str.strip()
)

unified_occupations["Occupation Name"] = (
    unified_occupations["Occupation Name"]
    .astype("string")
    .str.strip()
)

# Remove invalid records
unified_occupations = unified_occupations.dropna(
    subset=["Occupation ID", "Occupation Name"]
)

unified_occupations = unified_occupations[
    (unified_occupations["Occupation ID"] != "") &
    (unified_occupations["Occupation Name"] != "")
]

# Remove exact duplicates only
unified_occupations = (
    unified_occupations.drop_duplicates()
    .reset_index(drop=True)
)

display(unified_occupations.head(15))

,Occupation ID,Occupation Name,Source
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,ESCO
1,http://data.europa.eu/esco/occupation/000e93a3...,metal drawing machine operator,ESCO
2,http://data.europa.eu/esco/occupation/0019b951...,precision device inspector,ESCO
3,http://data.europa.eu/esco/occupation/0022f466...,air traffic safety technician,ESCO
4,http://data.europa.eu/esco/occupation/002da35b...,hospitality revenue manager,ESCO
5,http://data.europa.eu/esco/occupation/0044c991...,medical laboratory assistant,ESCO
6,http://data.europa.eu/esco/occupation/00634fc4...,asphalt laboratory technician,ESCO
7,http://data.europa.eu/esco/occupation/00674f21...,primary school teaching assistant,ESCO
8,http://data.europa.eu/esco/occupation/006cc1f9...,physiotherapist,ESCO
9,http://data.europa.eu/esco/occupation/00747307...,performing arts theatre instructor,ESCO


In [29]:
print("UNIFIED OCCUPATION DATABASE")
print("=" * 45)

print("Total occupations:", len(unified_occupations))

print("\nRecords by source:")
display(unified_occupations["Source"].value_counts())

print("\nUnique occupation names:",
      unified_occupations["Occupation Name"].nunique())

print("\nDuplicate records:",
      unified_occupations.duplicated().sum())

UNIFIED OCCUPATION DATABASE
Total occupations: 4055

Records by source:


Source
ESCO     3039
O*NET    1016
Name: count, dtype: int64


Unique occupation names: 4055

Duplicate records: 0


In [30]:
esco_skills_unified = esco_skills_clean[
    ["conceptUri", "preferredLabel"]
].copy()

esco_skills_unified = esco_skills_unified.rename(
    columns={
        "conceptUri": "Skill ID",
        "preferredLabel": "Skill Name"
    }
)

esco_skills_unified["Source"] = "ESCO"

In [31]:
onet_skills_unified = onet_skills_clean[
    ["Element ID", "Element Name"]
].copy()

onet_skills_unified = onet_skills_unified.rename(
    columns={
        "Element ID": "Skill ID",
        "Element Name": "Skill Name"
    }
)

onet_skills_unified["Source"] = "O*NET"

In [32]:
unified_skills = pd.concat(
    [esco_skills_unified, onet_skills_unified],
    ignore_index=True
)

# Standardize text fields
for column in ["Skill ID", "Skill Name"]:
    unified_skills[column] = (
        unified_skills[column]
        .astype("string")
        .str.strip()
    )

# Remove invalid records
unified_skills = unified_skills.dropna(
    subset=["Skill ID", "Skill Name"]
)

unified_skills = unified_skills[
    (unified_skills["Skill ID"] != "") &
    (unified_skills["Skill Name"] != "")
]

# Remove exact duplicate records
unified_skills = (
    unified_skills.drop_duplicates()
    .reset_index(drop=True)
)

display(unified_skills.head(15))

,Skill ID,Skill Name,Source
0,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,ESCO
1,http://data.europa.eu/esco/skill/00064735-8fad...,supervise correctional procedures,ESCO
2,http://data.europa.eu/esco/skill/000709ed-2be5...,apply anti-oppressive practices,ESCO
3,http://data.europa.eu/esco/skill/0007bdc2-dd15...,control compliance of railway vehicles regulat...,ESCO
4,http://data.europa.eu/esco/skill/00090cc1-1f27...,identify available services,ESCO
5,http://data.europa.eu/esco/skill/000bb1e4-89f0...,perform toxicological studies,ESCO
6,http://data.europa.eu/esco/skill/000c94d2-2a2e...,ensure coquille uniformity,ESCO
7,http://data.europa.eu/esco/skill/000f1d3d-220f...,Haskell,ESCO
8,http://data.europa.eu/esco/skill/001115fb-569f...,show initiative,ESCO
9,http://data.europa.eu/esco/skill/001d46db-035e...,train staff to reduce food waste,ESCO


In [33]:
print("UNIFIED SKILLS DATABASE")
print("=" * 45)

print("Total skill records:", len(unified_skills))

print("\nRecords by source:")
display(unified_skills["Source"].value_counts())

print("\nUnique skill names:",
      unified_skills["Skill Name"].nunique())

print("\nDuplicate records:",
      unified_skills.duplicated().sum())

UNIFIED SKILLS DATABASE
Total skill records: 13949

Records by source:


Source
ESCO     13939
O*NET       10
Name: count, dtype: int64


Unique skill names: 13949

Duplicate records: 0


In [34]:
print("OCCUPATION DATABASE VALIDATION")
print("=" * 45)

print("Total records:", len(unified_occupations))

print("Missing occupation IDs:",
      unified_occupations["Occupation ID"].isna().sum())

print("Missing occupation names:",
      unified_occupations["Occupation Name"].isna().sum())

print("Duplicate complete records:",
      unified_occupations.duplicated().sum())

print("\nRecords by source:")
display(unified_occupations["Source"].value_counts())

print("\nRepeated occupation IDs:")
display(
    unified_occupations[
        unified_occupations.duplicated(
            subset=["Source", "Occupation ID"],
            keep=False
        )
    ].head(10)
)

OCCUPATION DATABASE VALIDATION
Total records: 4055
Missing occupation IDs: 0
Missing occupation names: 0
Duplicate complete records: 0

Records by source:


Source
ESCO     3039
O*NET    1016
Name: count, dtype: int64


Repeated occupation IDs:


,Occupation ID,Occupation Name,Source


In [35]:
print("SKILL DATABASE VALIDATION")
print("=" * 45)

print("Total records:", len(unified_skills))

print("Missing skill IDs:",
      unified_skills["Skill ID"].isna().sum())

print("Missing skill names:",
      unified_skills["Skill Name"].isna().sum())

print("Duplicate complete records:",
      unified_skills.duplicated().sum())

print("\nRecords by source:")
display(unified_skills["Source"].value_counts())

print("\nRepeated skill IDs:")
display(
    unified_skills[
        unified_skills.duplicated(
            subset=["Source", "Skill ID"],
            keep=False
        )
    ].head(10)
)

SKILL DATABASE VALIDATION
Total records: 13949
Missing skill IDs: 0
Missing skill names: 0
Duplicate complete records: 0

Records by source:


Source
ESCO     13939
O*NET       10
Name: count, dtype: int64


Repeated skill IDs:


,Skill ID,Skill Name,Source


In [36]:
validation_summary = pd.DataFrame([
    {
        "Dataset": "Unified Occupations",
        "Total Records": len(unified_occupations),
        "Unique IDs": unified_occupations.groupby(
            "Source"
        )["Occupation ID"].nunique().sum(),
        "Missing IDs": unified_occupations[
            "Occupation ID"
        ].isna().sum(),
        "Duplicate Rows": unified_occupations.duplicated().sum()
    },
    {
        "Dataset": "Unified Skills",
        "Total Records": len(unified_skills),
        "Unique IDs": unified_skills.groupby(
            "Source"
        )["Skill ID"].nunique().sum(),
        "Missing IDs": unified_skills["Skill ID"].isna().sum(),
        "Duplicate Rows": unified_skills.duplicated().sum()
    }
])

display(validation_summary)

,Dataset,Total Records,Unique IDs,Missing IDs,Duplicate Rows
0,Unified Occupations,4055,4055,0,0
1,Unified Skills,13949,13949,0,0


In [37]:
from pathlib import Path

PROCESSED_DATA_PATH = (
    Path.cwd().parent / "data" / "processed"
)

PROCESSED_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:", PROCESSED_DATA_PATH)

Output directory: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed


In [38]:
# Save unified occupation database
unified_occupations.to_csv(
    PROCESSED_DATA_PATH / "unified_occupations.csv",
    index=False
)

# Save unified skills database
unified_skills.to_csv(
    PROCESSED_DATA_PATH / "unified_skills.csv",
    index=False
)

# Save ESCO occupation-skill relationships
esco_relations_clean.to_csv(
    PROCESSED_DATA_PATH / "esco_occupation_skill_relationships.csv",
    index=False
)

# Save O*NET occupation-skill relationships
onet_relationships.to_csv(
    PROCESSED_DATA_PATH / "onet_occupation_skill_relationships.csv",
    index=False
)

# Save normalized O*NET job titles
onet_job_titles_clean.to_csv(
    PROCESSED_DATA_PATH / "normalized_onet_job_titles.csv",
    index=False
)

print("All processed datasets saved successfully!")

All processed datasets saved successfully!


In [39]:
saved_files = [
    "unified_occupations.csv",
    "unified_skills.csv",
    "esco_occupation_skill_relationships.csv",
    "onet_occupation_skill_relationships.csv",
    "normalized_onet_job_titles.csv"
]

for filename in saved_files:
    file_path = PROCESSED_DATA_PATH / filename

    if file_path.exists():
        print(
            f"{filename} | "
            f"{file_path.stat().st_size / 1024:.2f} KB"
        )
    else:
        print(f"MISSING: {filename}")

unified_occupations.csv | 361.62 KB
unified_skills.csv | 1405.78 KB
esco_occupation_skill_relationships.csv | 27207.59 KB
onet_occupation_skill_relationships.csv | 1494.39 KB
normalized_onet_job_titles.csv | 5530.10 KB


In [40]:
# Day 3 data quality report

day3_report = pd.DataFrame([
    {
        "Dataset": "Unified Occupations",
        "Records": len(unified_occupations),
        "Columns": len(unified_occupations.columns),
        "Missing Cells": unified_occupations.isna().sum().sum(),
        "Duplicate Rows": unified_occupations.duplicated().sum()
    },
    {
        "Dataset": "Unified Skills",
        "Records": len(unified_skills),
        "Columns": len(unified_skills.columns),
        "Missing Cells": unified_skills.isna().sum().sum(),
        "Duplicate Rows": unified_skills.duplicated().sum()
    },
    {
        "Dataset": "ESCO Relationships",
        "Records": len(esco_relations_clean),
        "Columns": len(esco_relations_clean.columns),
        "Missing Cells": esco_relations_clean.isna().sum().sum(),
        "Duplicate Rows": esco_relations_clean.duplicated().sum()
    },
    {
        "Dataset": "O*NET Relationships",
        "Records": len(onet_relationships),
        "Columns": len(onet_relationships.columns),
        "Missing Cells": onet_relationships.isna().sum().sum(),
        "Duplicate Rows": onet_relationships.duplicated().sum()
    },
    {
        "Dataset": "O*NET Job Titles",
        "Records": len(onet_job_titles_clean),
        "Columns": len(onet_job_titles_clean.columns),
        "Missing Cells": onet_job_titles_clean.isna().sum().sum(),
        "Duplicate Rows": onet_job_titles_clean.duplicated().sum()
    }
])

display(day3_report)

# Save report
day3_report.to_csv(
    PROCESSED_DATA_PATH / "day3_data_quality_report.csv",
    index=False
)

print("\nDay 3 data quality report saved successfully!")

,Dataset,Records,Columns,Missing Cells,Duplicate Rows
0,Unified Occupations,4055,3,0,0
1,Unified Skills,13949,3,0,0
2,ESCO Relationships,126051,6,59,0
3,O*NET Relationships,18200,7,0,0
4,O*NET Job Titles,54269,6,49814,0



Day 3 data quality report saved successfully!


In [41]:
print("O*NET JOB TITLES — MISSING VALUES")
display(onet_job_titles_clean.isnull().sum())

print("\nESCO RELATIONSHIPS — MISSING VALUES")
display(esco_relations_clean.isnull().sum())

print("\nO*NET JOB TITLE ESSENTIAL FIELD CHECK")

print(
    "Missing occupation codes:",
    onet_job_titles_clean["O*NET-SOC Code"].isna().sum()
)

print(
    "Missing job titles:",
    onet_job_titles_clean["Job Title"].isna().sum()
)

print("\nESCO ESSENTIAL FIELD CHECK")

print(
    "Missing occupation URIs:",
    esco_relations_clean["occupationUri"].isna().sum()
)

print(
    "Missing skill URIs:",
    esco_relations_clean["skillUri"].isna().sum()
)

O*NET JOB TITLES — MISSING VALUES


O*NET-SOC Code              0
Title                       0
Job Title                   0
Short Title             49814
Source(s)                   0
Normalized Job Title        0
dtype: int64


ESCO RELATIONSHIPS — MISSING VALUES


occupationUri       0
occupationLabel     0
relationType        0
skillType          59
skillUri            0
skillLabel          0
dtype: int64


O*NET JOB TITLE ESSENTIAL FIELD CHECK
Missing occupation codes: 0
Missing job titles: 0

ESCO ESSENTIAL FIELD CHECK
Missing occupation URIs: 0
Missing skill URIs: 0
